<p class="lead">Omics data break the habits acquired in the rest of this course. There are more variables than samples — five thousand genes measured in a hundred and thirty-eight tissue pieces — so every analysis is a multiple-comparison problem before it is anything else; the samples are not independent, because two of them come from the same patient; and the observations themselves are related by evolution, so two protein sequences in a training set can be the same protein read in two species. This unit works through the complete path on two frozen datasets the group uses: 621 reviewed UniProt sequences from six protein families, where a random train/test split quietly hides homology leakage, and the periodontitis expression matrix, from quality control through a paired <code>limma</code> model and false-discovery-rate control to over-representation and gene-set enrichment analysis, finishing with the four ways an omics analysis most often goes wrong.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad recorre el análisis de datos de secuencias y de expresión génica. Primero se manipulan 621 secuencias reales de proteínas revisadas de UniProt, repartidas en seis familias (globinas, proteasas de serina S1, lisozima C, anhidrasas carbónicas, quinasas AGC y citocromos P450) con <code>seqinr</code>: lectura de FASTA, composición de aminoácidos, propiedades fisicoquímicas y vectores de k-meros. Un clasificador de familias alcanza 0,937 de exactitud con partición aleatoria y 0,905 cuando los grupos de secuencias homólogas se mantienen enteros en un solo pliegue (0,879 agrupando por nombre de proteína); el diagnóstico decisivo es que con partición aleatoria el 21,9 % de las secuencias de prueba tiene un pariente casi idéntico en el conjunto de entrenamiento, y con partición por grupos ese porcentaje baja a cero. Luego se analiza la matriz de expresión congelada del estudio de periodontitis (5.000 genes x 138 muestras, 69 pares enfermo/sano del mismo paciente): control de calidad, PCA, un modelo lineal pareado ajustado con <code>limma</code> —la herramienta correcta para intensidades log2 de microarreglo— que reproduce exactamente la prueba t pareada antes de la moderación bayesiana empírica, control de la tasa de falsos descubrimientos con Benjamini-Hochberg (3.771 genes con FDR &lt; 0,05) y umbrales de tamaño de efecto. Se construyen los gráficos habituales (volcán, MA, mapa de calor) y se interpretan los resultados con análisis de sobre-representación hipergeométrico escrito a mano y con <code>fgsea</code>. La unidad termina con los errores característicos de la ómica: p mucho mayor que n, esquemas de permutación que ignoran el diseño pareado, doble uso de los datos, la maldición del ganador y los límites de una sola cohorte sin covariables demográficas.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Handle sequence data programmatically with <code>seqinr</code>: parsing, composition, physico-chemical descriptors and k-mer features, and build a protein-family classifier evaluated with a group split.</li>
<li>Normalise and explore an expression matrix (distributions, library-size proxies, PCA, structure that is not biology) and run a differential-expression analysis with correct multiple-testing control.</li>
<li>Perform and interpret over-representation and gene-set enrichment analysis, and present results (volcano, MA, heatmap, enrichment plots) without the usual misreadings.</li>
<li>Recognise the specific pitfalls of omics: p much greater than n, homology and batch structure, double dipping, circular enrichment, the winner's curse and the limits of a single cohort.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read it linearly with the code running; every number in the prose is printed by a chunk of this document. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","seqinr","ranger","nnet","pheatmap","patchwork"))</code> and <code>BiocManager::install(c("limma","fgsea"))</code> once before the first chunk. The analysis uses <code>seqinr</code><sup class="c"><a href="#r1">[1]</a></sup> for sequences, <code>limma</code><sup class="c"><a href="#r4">[4]</a></sup> for the differential-expression model, <code>fgsea</code><sup class="c"><a href="#r30">[30]</a></sup> for the enrichment, and base R<sup class="c"><a href="#r28">[28]</a></sup> with <code>ranger</code><sup class="c"><a href="#r31">[31]</a></sup> and <code>ggplot2</code><sup class="c"><a href="#r32">[32]</a></sup> for the rest. This is an <b>extended</b> unit: students whose project does not involve expression data can read sections 1-2 and stop. It builds on <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a> (multiple comparisons), <a href="../U9_Linear_Models/index.html">Unit 9</a> (linear models), <a href="../U12_Validation/index.html">Unit 12</a> (group splits and leakage) and <a href="../U13_Features_Dimensionality/index.html">Unit 13</a> (PCA). The whole document knits in about three minutes.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(seqinr);  library(ranger); library(nnet)
  library(limma);      library(fgsea);   library(ggplot2); library(patchwork); library(pheatmap)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10), panel.grid.minor = element_blank()))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| seqinr", as.character(packageVersion("seqinr")),
    "| limma", as.character(packageVersion("limma")),
    "| fgsea", as.character(packageVersion("fgsea")),
    "| ranger", as.character(packageVersion("ranger")), "\n")



## 1. Two kinds of omics data

<div class="box bridge"><h4>Domain context</h4>
<p>A <b>gene</b> is a stretch of DNA transcribed into messenger RNA and usually translated into a <b>protein</b>, a
chain of twenty kinds of amino acid written as letters (M for methionine, W for tryptophan, and so on). A
<b>protein family</b> groups proteins that descend from a common ancestor and share a fold and a function;
<b>orthologues</b> are the same protein in different species (human and mouse myoglobin) and <b>paralogues</b> are
copies within one genome that diverged after duplication (haemoglobin alpha and beta). Both are <b>homologues</b>,
and both make two rows of a sequence table statistically dependent. <b>UniProt</b><sup class="c"><a href="#r33">[33]</a></sup>
is the reference protein knowledgebase; its <i>reviewed</i> (Swiss-Prot) section is manually curated and carries a
family annotation, which is where the labels used here come from. The <b>transcriptome</b> is the set of mRNA
abundances in a tissue sample: a <b>microarray</b> measures them as fluorescence intensities on probes, and RNA
sequencing measures them as read counts. The expression data used here come from a microarray study of
<b>periodontitis</b>, a chronic inflammatory disease of the tissues around the teeth, in which each patient
contributed one diseased gingival site and one clinically healthy site — a <b>paired design</b>. The values are
<b>log2 RMA intensities</b><sup class="c"><a href="#r17">[17]</a></sup>, so a difference of 1 means a two-fold change
in abundance. A <b>gene set</b> is a curated list of genes sharing an annotation — a KEGG
pathway<sup class="c"><a href="#r20">[20]</a></sup> or a Gene Ontology biological
process<sup class="c"><a href="#r18">[18]</a></sup><sup class="c"><a href="#r19">[19]</a></sup> — and
<b>enrichment</b> asks whether the genes a study highlights fall into such a list more often than chance.</p></div>

Both datasets are frozen with the course. `protein_families.csv` and `protein_families.fasta` hold **621 reviewed
UniProt sequences** in six families, and `geo_periodontitis_expr.csv` is the slice of GEO series
GSE16134<sup class="c"><a href="#r13">[13]</a></sup> studied by the group's collaborators<sup class="c"><a href="#r14">[14]</a></sup><sup class="c"><a href="#r15">[15]</a></sup><sup class="c"><a href="#r16">[16]</a></sup>:
the 5,000 most variable genes across 138 samples, 69 patients × (one diseased site + one healthy site). That series
carries **no demographic covariates at all** — no age, no sex, no smoking status — which is itself a lesson about
public data and is the reason the structure modelled below is the patient rather than a batch.



In [ ]:
dp <- function(f) file.path("..", "data", f)      # knitr runs with cwd = this unit's folder
AAs <- strsplit("ACDEFGHIKLMNPQRSTVWY", "")[[1]]
fam <- fread(dp("protein_families.csv"))
fa  <- read.fasta(dp("protein_families.fasta"), seqtype = "AA", as.string = FALSE)
cat(sprintf("table: %d x %d | FASTA records parsed with seqinr::read.fasta: %d\n", nrow(fam), ncol(fam), length(fa)))
cat("header of the first record:", substr(attr(fa[[1]], "Annot"), 1, 78), "\n")
cat(sprintf("the FASTA lengths agree with the table for all %d entries: %s\n",
            nrow(fam), all(lengths(fa) == fam$length)))
knitr::kable(fam[, .(n = .N, mean_length = round(mean(length)), min = min(length), max = max(length),
                     organisms = uniqueN(organism), protein_names = uniqueN(protein_name)),
                 by = family][order(mean_length)])



Six families of unequal size — globin, serine protease S1 and lysozyme C with 118 sequences each, carbonic anhydrase
108, AGC protein kinase 99 and cytochrome P450 60 — spanning 148 distinct organisms and 411 distinct protein names.
The length column already shows how easy the classification task will be: lysozyme C averages 145 residues and the AGC
kinases 634, a factor of four.



In [ ]:
KD <- c(A = 1.8, R = -4.5, N = -3.5, D = -3.5, C = 2.5, Q = -3.5, E = -3.5, G = -0.4, H = -3.2, I = 4.5,
        L = 3.8, K = -3.9, M = 1.9, F = 2.8, P = -1.6, S = -0.8, T = -0.7, W = -0.9, Y = -1.3, V = 4.2)
nonstd <- sapply(fam$sequence, function(s) any(!(strsplit(s, "")[[1]] %in% AAs)))
props <- rbindlist(lapply(which(!nonstd), function(i) {
  ch <- strsplit(fam$sequence[i], "")[[1]]
  data.table(family = fam$family[i], length = length(ch), kDa = pmw(ch) / 1000, pI = computePI(ch),
             gravy = mean(KD[ch]), aromaticity = mean(ch %in% c("F", "W", "Y")))
}))
cat(sprintf("sequences containing non-standard residues (X, B, Z): %d of %d - excluded from the descriptors\n",
            sum(nonstd), nrow(fam)))
knitr::kable(props[, lapply(.SD, mean), by = family][order(length)], digits = 2)



<div class="box bridge"><h4>Substitution — <code>seqinr</code> instead of Biopython's ProtParam</h4>
<p>The Python edition computes molecular weight, isoelectric point, GRAVY and aromaticity with
<code>Bio.SeqUtils.ProtParam</code>. R has no Biopython; <code>seqinr</code> provides <code>pmw()</code> for the
molecular weight and <code>computePI()</code> for the isoelectric point with the same pK tables, while GRAVY is the
mean Kyte-Doolittle hydropathy (the twenty constants are written out above, so nothing is hidden) and aromaticity is
the fraction of F, W and Y residues — Biopython's own definitions. Like ProtParam, the calculation refuses ambiguous
letters, so the six sequences containing X, B or Z are excluded from the descriptor table and kept everywhere
else.</p></div>

The descriptors separate the families about as well as the length does: cytochrome P450 is the most basic family
(mean pI 8.35), carbonic anhydrase and the AGC kinases the most hydrophilic (GRAVY -0.48 and -0.47), and the
aromaticity hardly varies at all (0.08-0.10), which is a useful reminder that a descriptor can be perfectly well
defined and carry no discriminating information.



In [ ]:
comp <- do.call(rbind, lapply(fam$sequence, function(s) {
  ch <- strsplit(s, "")[[1]]; tabulate(match(ch, AAs), 20) / length(ch)
}))
colnames(comp) <- AAs
cdt <- as.data.table(comp)[, family := fam$family]
knitr::kable(cdt[, lapply(.SD, mean), by = family, .SDcols = c("C", "W", "G", "K", "L", "P")], digits = 4)
cat(sprintf("cysteine fraction: lysozyme C %.4f vs all other families %.4f\n",
            mean(comp[fam$family == "lysozyme_C", "C"]), mean(comp[fam$family != "lysozyme_C", "C"])))
lyso <- fam[grepl("^Lysozyme C ", protein_name)]
cat(sprintf("'Lysozyme C' appears %d times, in %d different organisms - e.g. %s\n",
            nrow(lyso), uniqueN(lyso$organism), paste(sub(" \\(.*", "", head(lyso$organism, 3)), collapse = ", ")))



Lysozyme C contains 6.1 % cysteine against 2.0 % in the other five families — the eight disulphide-forming cysteines of
a 129-residue enzyme — so a classifier can reach high accuracy on composition alone. The last line is the problem this
section is about: the same protein, "Lysozyme C", appears 46 times, once per organism. Those 46 rows are not 46
independent observations of what a lysozyme looks like; they are one observation read 46 times, with mutations.



In [ ]:
cm <- melt(cdt, id.vars = "family", variable.name = "aa", value.name = "frac")[
  , .(frac = mean(frac)), by = .(family, aa)]
cm[, z := (frac - mean(frac)) / sd(frac), by = aa]
p0 <- ggplot(cm, aes(aa, family, fill = z)) + geom_tile() +
  scale_fill_gradient2(low = PAL[["blue"]], mid = "white", high = PAL[["orange"]], name = "z-score\nacross families") +
  labs(title = "Mean amino-acid composition, standardised within each residue", x = NULL, y = NULL)
print(p0)



<p class="caption"><b>Figure 1.</b> Mean composition of each family, standardised within each amino acid so that the
colour shows which family is unusual for that residue. Lysozyme C (top row of the orange block at C and W) and the
serine proteases stand out in cysteine; the AGC kinases are rich in lysine and arginine and poor in tryptophan. The
pattern is systematic enough that a linear classifier on composition alone reaches high accuracy — and systematic
*because* the families are evolutionary groups, which is exactly why the split protocol matters.</p>

## 2. Why a random split flatters a sequence classifier

The numeric representation used from here on is the **dipeptide profile**: the relative frequency of each of the
\(20^2 = 400\) ordered amino-acid pairs, L2-normalised so that two profiles can be compared by their dot product
(the cosine similarity). A second representation, the 3-mer profile, is used only to measure how similar two
sequences are — a cheap stand-in for a sequence-identity calculation.



In [ ]:
KMERS <- as.vector(outer(AAs, AAs, paste0))
kmer_features <- function(s, k = 2) {            # relative frequency of each dipeptide
  n <- nchar(s) - k + 1
  tabulate(match(substring(s, 1:n, k:(n + k - 1)), KMERS), length(KMERS)) / n
}
Xk <- t(sapply(fam$sequence, kmer_features)); rownames(Xk) <- NULL
Xk <- Xk / sqrt(rowSums(Xk^2))                   # L2 normalisation: dot product = cosine similarity
y  <- factor(fam$family)
tri <- lapply(fam$sequence, function(s) { n <- nchar(s) - 2; table(substring(s, 1:n, 3:(n + 2))) })
vocab <- unique(unlist(lapply(tri, names)))
M <- matrix(0, nrow(fam), length(vocab))
for (i in seq_along(tri)) M[i, match(names(tri[[i]]), vocab)] <- as.numeric(tri[[i]])
Mn <- M / sqrt(rowSums(M^2)); SIM <- tcrossprod(Mn); diag(SIM) <- 1
cat(sprintf("dipeptide matrix %d x %d | 3-mer vocabulary %d | mean pairwise 3-mer similarity %.3f\n",
            nrow(Xk), ncol(Xk), length(vocab), mean(SIM[upper.tri(SIM)])))
ly <- which(grepl("^Lysozyme C ", fam$protein_name)); S2 <- SIM[ly, ly]
cat(sprintf("mean similarity among the %d 'Lysozyme C' entries: %.3f (five times the background)\n",
            length(ly), mean(S2[upper.tri(S2)])))


In [ ]:
hcl    <- hclust(as.dist(pmax(1 - SIM, 0)), method = "average")
groups <- cutree(hcl, h = 0.5)                   # homology groups at cosine distance 0.5
cat(sprintf("homology groups at cosine distance 0.5: %d groups, largest %d members, %d singletons\n",
            length(unique(groups)), max(table(groups)), sum(table(groups) == 1)))
cat(sprintf("for comparison, grouping by protein name gives %d groups\n", uniqueN(fam$protein_name)))
strat_folds <- function(y, k = 5, seed = 0) {    # stratified k-fold, written out
  set.seed(seed); f <- integer(length(y))
  for (lv in levels(y)) { i <- which(y == lv); f[i] <- sample(rep(1:k, length.out = length(i))) }
  f
}
group_folds <- function(g, k = 5) {              # greedy balanced assignment of whole groups to folds
  sz <- sort(table(g), decreasing = TRUE); load <- integer(k); gf <- integer(length(sz))
  for (i in seq_along(sz)) { j <- which.min(load); gf[i] <- j; load[j] <- load[j] + sz[i] }
  names(gf) <- names(sz); gf[as.character(g)]
}
f_rand <- strat_folds(y); f_name <- group_folds(fam$protein_name); f_hom <- group_folds(groups)
cat("fold sizes - random:", table(f_rand), "| homology:", table(f_hom), "\n")



The 621 sequences fall into 345 homology groups, the largest with 33 members and 256 of them singletons. Grouping by
protein name gives 411 groups, a finer and stricter partition. Both are proxies for what a real protocol would do —
cluster with CD-HIT<sup class="c"><a href="#r34">[34]</a></sup> or use the pre-computed UniRef
clusters<sup class="c"><a href="#r35">[35]</a></sup> at a stated identity threshold — and both must be reported for
what they are. One constraint is absolute: the grouping variable can never be the class label, because a fold
containing no globin at all cannot be scored on globins.



In [ ]:
knn1_cos <- function(Xtr, ytr, Xte) ytr[max.col(tcrossprod(Xte, Xtr), ties.method = "first")]
cv_acc <- function(model, folds) mean(sapply(sort(unique(folds)), function(f) {
  tr <- folds != f
  pred <- switch(model,
    knn    = knn1_cos(Xk[tr, ], y[tr], Xk[!tr, ]),
    logreg = predict(nnet::multinom(y ~ ., data = data.frame(Xk[tr, ], y = y[tr]),
                                    trace = FALSE, maxit = 300, MaxNWts = 5000),
                     data.frame(Xk[!tr, ])),
    rf     = predict(ranger(y ~ ., data = data.frame(Xk[tr, ], y = y[tr]), num.trees = 300,
                            seed = 0, num.threads = 2), data.frame(Xk[!tr, ]))$predictions)
  mean(pred == y[!tr])
}))
acc_tab <- rbindlist(lapply(c(knn = "knn", logreg = "logreg", rf = "rf"), function(mo)
  data.table(model = mo, random_split = cv_acc(mo, f_rand),
             by_protein_name = cv_acc(mo, f_name), by_homology_cluster = cv_acc(mo, f_hom))))
knitr::kable(acc_tab, digits = 3)
cat(sprintf("majority-class baseline %.3f\n", max(table(y)) / length(y)))



Three models, three protocols, one data set. Every model loses accuracy when homologous sequences are kept together,
and the loss depends on how much the model relies on recognising individual sequences: the nearest-neighbour
classifier falls from **0.937** to **0.905** (homology clusters) and to **0.879** (protein names), while multinomial
logistic regression on the same features falls only from 0.990 to 0.981, and the random forest from 0.990 to 0.982.
That asymmetry is itself the lesson. A model that generalises over composition barely notices the change; a model
whose prediction *is* "find the most similar training sequence" — the machine-learning version of a BLAST search —
loses most of what a random split gave it.

The accuracy gap understates the problem, because this six-family task is easy. The honest diagnostic is not the
accuracy drop but the *composition of the test set*: how similar is each test sequence to the nearest sequence the
model was allowed to train on?



In [ ]:
max_sim_to_train <- function(folds) unlist(lapply(sort(unique(folds)), function(f)
  apply(SIM[folds == f, folds != f, drop = FALSE], 1, max)))
ms <- list(`random split` = max_sim_to_train(f_rand),
           `protein-name split` = max_sim_to_train(f_name),
           `homology-group split` = max_sim_to_train(f_hom))
for (nm in names(ms))
  cat(sprintf("%-22s median %.3f | at least 0.9 similar: %4.1f %% | at least 0.7: %4.1f %%\n",
              nm, median(ms[[nm]]), 100 * mean(ms[[nm]] >= 0.9), 100 * mean(ms[[nm]] >= 0.7)))



Under a random split the median test sequence has a nearest training relative at similarity 0.596, 21.9 % of test
sequences have one above 0.9 and 42.0 % above 0.7. Under the homology-group split the median falls to 0.315 and *no*
test sequence has a relative above 0.7 — the orthologues have been removed from the training set, which is what a
group split is for. The protein-name split sits in between (median 0.449, 11.0 % above 0.9), which is why it gives the
lowest nearest-neighbour accuracy of the three: it separates the 46 lysozymes from each other but leaves other close
relatives behind.



In [ ]:
set.seed(0)
curve <- rbindlist(lapply(c(30, 60, 120, 240), function(n_train) {
  ar <- c(); ag <- c()
  for (rep in 1:20) {
    idx <- sample(nrow(Xk), n_train)
    if (length(unique(y[idx])) == nlevels(y))
      ar <- c(ar, mean(knn1_cos(Xk[idx, ], y[idx], Xk[-idx, ]) == y[-idx]))
    keep <- integer(0)
    for (cl in sample(unique(groups))) { keep <- c(keep, which(groups == cl)); if (length(keep) >= n_train) break }
    keep <- keep[1:n_train]
    if (length(unique(y[keep])) < nlevels(y)) next
    held <- which(!(groups %in% unique(groups[keep])))
    ag <- c(ag, mean(knn1_cos(Xk[keep, ], y[keep], Xk[held, ]) == y[held]))
  }
  data.table(n_train = n_train, random = mean(ar), homology_disjoint = mean(ag),
             gap = mean(ar) - mean(ag), usable_repeats = length(ag))
}))
knitr::kable(curve, digits = 3)



The penalty of the honest protocol grows as the training set shrinks: with 240 training sequences the
nearest-neighbour gap is 0.071, with 30 it is 0.120 — and 30 sequences is the realistic size of a thesis project's
curated set. (The `usable_repeats` column records that only 12 of 20 repeats at n = 30 contained all six families and
could be scored at all, which is itself part of the cost of grouping.)



In [ ]:
pcs <- prcomp(Xk, center = TRUE)$x
ps1 <- ggplot(data.table(PC1 = pcs[, 1], PC2 = pcs[, 2], family = fam$family),
              aes(PC1, PC2, colour = family)) + geom_point(size = 0.9, alpha = 0.8) +
  scale_colour_manual(values = c(unname(PAL), "#999999", "#111111"), name = NULL) +
  guides(colour = guide_legend(override.aes = list(size = 1.8))) +
  labs(title = "Dipeptide space: families do separate") +
  theme(legend.position = "bottom", legend.text = element_text(size = 5.4),
        legend.key.height = unit(6, "pt"))
msdf <- rbindlist(lapply(names(ms)[c(1, 3)], function(nm) data.table(split = nm, v = ms[[nm]])))
ps2 <- ggplot(msdf, aes(v, fill = split)) +
  geom_histogram(aes(y = after_stat(density)), bins = 30, position = "identity", alpha = 0.6) +
  scale_fill_manual(values = c(`random split` = PAL[["orange"]], `homology-group split` = PAL[["blue"]]), name = NULL) +
  labs(title = "What the random split leaves behind",
       x = "similarity to the nearest training sequence", y = "density") +
  theme(legend.position = "bottom", legend.text = element_text(size = 6))
amelt <- melt(acc_tab, id.vars = "model", variable.name = "protocol", value.name = "acc")
ps3 <- ggplot(amelt, aes(model, acc, fill = protocol)) + geom_col(position = "dodge", width = 0.75) +
  coord_cartesian(ylim = c(0.8, 1)) +
  scale_fill_manual(values = unname(PAL[c("orange", "green", "blue")]), name = NULL) +
  labs(title = "Same features, three protocols", x = NULL, y = "cross-validated accuracy") +
  theme(legend.position = "bottom", legend.text = element_text(size = 5.6))
print(ps1 | ps2 | ps3)



<p class="caption"><b>Figure 2.</b> Left: the first two principal components of the L2-normalised dipeptide profiles;
the six families occupy distinguishable regions, which is why every classifier scores well. Centre: the diagnostic
that matters — the random split (orange) leaves a near-identical relative in the training set for a fifth of the test
sequences, the homology split (blue) leaves none above 0.7. Right: the accuracy consequence, large for the
nearest-neighbour model (0.937 to 0.879) and small for logistic regression and the random forest, whose bars barely
move between protocols.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>Re-cut the homology groups at cosine distance 0.3 and at 0.7 (change <code>h =</code> in the <code>cutree</code>
call) and repeat the three-model comparison. How do the number of groups and the grouped accuracy move, and which cut
would you defend to a reviewer who asks "at what identity did you cluster"? Then repeat the comparison using only the
20 composition features (<code>comp</code>) instead of the 400 dipeptides: does the simpler feature set suffer more or
less from homology leakage, and why?</p></div>

## 3. The expression matrix: quality control and structure



In [ ]:
ex   <- fread(dp("geo_periodontitis_expr.csv"))
E    <- as.matrix(ex[, -1]); rownames(E) <- ex[[1]]
meta <- fread(dp("geo_periodontitis_meta.csv")); gi <- fread(dp("geo_periodontitis_genes.csv"))
cols <- colnames(E)
patients <- sort(unique(substr(cols, 1, nchar(cols) - 2)))
status   <- as.integer(grepl("_D$", cols))                 # 1 = diseased site
cs <- colSums(E)
cat(sprintf("expression matrix: %d genes x %d samples | patients: %d | values %.2f to %.2f (log2 RMA)\n",
            nrow(E), ncol(E), length(patients), min(E), max(E)))
cat("samples per status:", paste(names(table(meta$status)), table(meta$status), collapse = ", "), "\n")
cat(sprintf("column sums (a library-size proxy for arrays): min %.0f, max %.0f, coefficient of variation %.4f\n",
            min(cs), max(cs), sd(cs) / mean(cs)))
cat("columns of the metadata table:", paste(names(meta), collapse = ", "), "\n")



The metadata carry a sample identifier, the patient, the status, the site label and the pairing flag — and nothing
else. There is no age, no sex, no smoking status, so no covariate can be adjusted for and no subgroup analysis is
possible. The library-size proxy varies by 2.2 %, which for RMA-summarised arrays is what one expects after quantile
normalisation.



In [ ]:
sub <- melt(as.data.table(E[, 1:24])[, gene := rownames(E)], id.vars = "gene",
            variable.name = "sample", value.name = "x")
pq1 <- ggplot(sub, aes(sample, x)) + geom_boxplot(outlier.shape = NA, fill = PAL[["blue"]], linewidth = 0.25) +
  labs(title = "Distributions after RMA: comparable", x = "first 24 samples", y = "log2 intensity") +
  theme(axis.text.x = element_blank())
pq2 <- ggplot(data.table(x = as.vector(E)), aes(x)) + geom_histogram(bins = 80, fill = PAL[["blue"]]) +
  labs(title = "Bimodal: silent and expressed genes", x = "log2 intensity", y = "gene x sample values")
pq3 <- ggplot(data.table(cs = cs, i = seq_along(cs), st = ifelse(status == 1, "diseased", "healthy")),
              aes(cs, i, colour = st)) + geom_point(size = 0.8) +
  scale_colour_manual(values = c(diseased = PAL[["orange"]], healthy = PAL[["blue"]]), name = NULL) +
  labs(title = "No systematic shift between site types", x = "column sum (library-size proxy)", y = "sample index") +
  theme(legend.position = "bottom", legend.text = element_text(size = 6))
print(pq1 | pq2 | pq3)



<p class="caption"><b>Figure 3.</b> Left: per-sample distributions of the first 24 arrays; RMA has already
quantile-normalised them, so medians and quartiles coincide and no further normalisation is applied. Centre: the
pooled histogram is bimodal, the lower mode being genes close to background. Right: the library-size proxy varies by
only 2.2 % and the diseased (orange) and healthy (blue) samples are interleaved, so there is no depth confounding of
the kind that count-based RNA-seq data would require normalising away.</p>

This is also where the choice of statistical machinery is made. These are **continuous, already normalised log2
intensities**, not counts. The negative-binomial machinery of DESeq2<sup class="c"><a href="#r2">[2]</a></sup> models
integer counts with a mean-variance relationship that microarray intensities do not have; applying it here would be a
category error, and the best practices of count-based pipelines<sup class="c"><a href="#r27">[27]</a></sup> do not
transfer directly. The correct tool is a linear model per gene with an empirical-Bayes moderated variance — the
`limma` approach<sup class="c"><a href="#r4">[4]</a></sup><sup class="c"><a href="#r5">[5]</a></sup> — and the paired
design enters as a patient term in the design matrix. The Python edition of this unit fits the same model with
ordinary least squares because no limma port exists there; in R we use limma itself, and section 4 shows exactly what
the moderation adds.



In [ ]:
pc  <- prcomp(t(E), center = TRUE, scale. = FALSE)
evr <- pc$sdev^2 / sum(pc$sdev^2); Ys <- pc$x
cat("variance explained by PC1..PC5:", round(evr[1:5], 3), "\n")
for (k in 1:3) {
  tt <- t.test(Ys[status == 1, k], Ys[status == 0, k])
  cat(sprintf("PC%d: diseased mean %+.2f, healthy %+.2f, t-test p = %.2e\n",
              k, mean(Ys[status == 1, k]), mean(Ys[status == 0, k]), tt$p.value))
}
cat(sprintf("correlation of PC1 with the library-size proxy: %.2f\n", cor(Ys[, 1], cs)))


In [ ]:
pdt <- data.table(PC1 = Ys[, 1], PC2 = Ys[, 2], sample = cols,
                  site = ifelse(status == 1, "diseased site", "healthy site"),
                  patient = substr(cols, 1, nchar(cols) - 2))
seg <- dcast(pdt[patient %in% patients[1:25]], patient ~ site, value.var = c("PC1", "PC2"))
pa1 <- ggplot(pdt, aes(PC1, PC2, colour = site)) +
  geom_segment(data = seg, aes(x = `PC1_diseased site`, y = `PC2_diseased site`,
                               xend = `PC1_healthy site`, yend = `PC2_healthy site`),
               inherit.aes = FALSE, colour = "grey45", linewidth = 0.2) +
  geom_point(size = 1.2) +
  scale_colour_manual(values = c(`diseased site` = PAL[["orange"]], `healthy site` = PAL[["blue"]]), name = NULL) +
  labs(title = "Samples in PCA space; lines join the two sites of a patient",
       x = sprintf("PC1 (%.0f %%)", 100 * evr[1]), y = sprintf("PC2 (%.0f %%)", 100 * evr[2])) +
  theme(legend.position = "bottom", legend.text = element_text(size = 6))
pa2 <- ggplot(data.table(k = 1:5, v = evr[1:5]), aes(factor(k), v)) +
  geom_col(fill = PAL[["blue"]], width = 0.7) +
  labs(title = "PC1 carries 27 % and tracks the phenotype",
       x = "principal component", y = "variance explained")
print(pa1 | pa2)



<p class="caption"><b>Figure 4.</b> PC1 explains 27 % of the total variance and separates diseased from healthy sites
(p = 4.3e-22 comparing PC1 scores between site types); PC2 explains 13 % and does not (p = 0.093). The thin lines join
the two samples of the same patient: most are short, meaning the two sites of one patient resemble each other in the
dimensions that PCA ranks highest — the structure that the paired analysis of the next section exploits.</p>

Finding the design in the first principal component is reassuring but not automatic. In many data sets PC1 is the
processing date, the RNA quality or the hospital, and the phenotype appears in PC4 if at all — the classic batch-effect
situation<sup class="c"><a href="#r9">[9]</a></sup> whose standard remedy, ComBat<sup class="c"><a href="#r10">[10]</a></sup>,
also removes real signal when the batch is confounded with the condition. Here the only grouping variable the series
provides is the patient, so the patient is what we model. PC1 correlates with the library-size proxy at only 0.16, so
it is not a technical axis in disguise.

## 4. Differential expression with a paired linear model

For one gene, write the expression of sample \(s\) as

\[ y_s = \mu + \beta\,\text{diseased}_s + \alpha_{\text{patient}(s)} + \varepsilon_s, \]

a linear model with one coefficient of interest, \(\beta\), and one intercept per patient. The patient terms absorb
everything that makes a person's gingiva different from another's — genotype, age, oral hygiene, the unmeasured
covariates this series does not provide — so \(\beta\) is estimated from *within-patient* contrasts only. For a design
with exactly two samples per patient this model is algebraically identical to a paired t-test on the differences, and
the next chunk demonstrates the identity rather than asserting it.

<div class="box bridge"><h4>Worked example: one gene, by hand</h4>
<p>Take <code>TNFRSF17</code>, a plasma-cell marker. For each of the 69 patients compute the difference
diseased - healthy; the first five are +2.834, +3.582, +0.403, -0.191 and +2.935 log2 units. The mean of all 69
differences is 2.1312 and their standard deviation is 1.7268, so the standard error is
1.7268/sqrt(69) = 0.2079 and the paired t statistic is 2.1312/0.2079 = 10.252 with 68 degrees of freedom, giving
p = 1.96e-15. The same number must come out of <code>t.test(..., paired = TRUE)</code> and out of the linear model
<code>lm(y ~ patient + status)</code> with its 69 patient intercepts. The unpaired model on the same gene gives
t = 8.250 instead, because it must compete against the between-patient variability that the pairing removes.</p></div>



In [ ]:
g  <- "TNFRSF17"
Dm <- E[, paste0(patients, "_D")]; Hm <- E[, paste0(patients, "_H")]
delta  <- Dm - Hm                                  # genes x patients, within-patient log2 fold changes
log2fc <- rowMeans(delta)
dv <- delta[g, ]
t_hand <- mean(dv) / (sd(dv) / sqrt(length(dv)))
cat(sprintf("by hand          : mean difference %.4f, sd %.4f, SE %.4f, t = %.3f, p = %.3e\n",
            mean(dv), sd(dv), sd(dv) / sqrt(69), t_hand, 2 * pt(-abs(t_hand), 68)))
tt <- t.test(Dm[g, ], Hm[g, ], paired = TRUE)
cat(sprintf("t.test(paired)   : t = %.3f, p = %.3e\n", tt$statistic, tt$p.value))
pat <- factor(substr(cols, 1, nchar(cols) - 2))
fit1 <- lm(E[g, ] ~ pat + status); s1 <- summary(fit1)$coefficients["status", ]
cat(sprintf("lm with patients : beta = %.4f, t = %.3f, p = %.3e, residual df = %d\n",
            s1[1], s1[3], s1[4], fit1$df.residual))
s2 <- summary(lm(E[g, ] ~ status))$coefficients["status", ]
cat(sprintf("lm, no patient   : beta = %.4f, t = %.3f, p = %.3e\n", s2[1], s2[3], s2[4]))



Four routes, one number: \(\beta = 2.1312\) with t = 10.252 and p = 1.96e-15 on 68 residual degrees of freedom. Dropping
the patient term leaves the coefficient untouched — the design is balanced — but inflates the standard error, and the
t statistic falls to 8.250.



In [ ]:
design <- model.matrix(~ pat + status)
fitl <- eBayes(lmFit(E, design))
tl   <- topTable(fitl, coef = "status", number = Inf, sort.by = "none")
sdd  <- apply(delta, 1, sd)
t_paired <- log2fc / (sdd / sqrt(69)); p_paired <- 2 * pt(-abs(t_paired), 68)
p_unpaired <- apply(E, 1, function(v) t.test(v[status == 1], v[status == 0], var.equal = TRUE)$p.value)
q_paired <- p.adjust(p_paired, "BH"); q_unpaired <- p.adjust(p_unpaired, "BH")
cat(sprintf("ordinary paired t-test : FDR < 0.05 for %d genes | unpaired: %d (of %d tested)\n",
            sum(q_paired < 0.05), sum(q_unpaired < 0.05), nrow(E)))
cat(sprintf("paired p smaller than unpaired p for %d of %d genes (%.0f %%)\n",
            sum(p_paired < p_unpaired), nrow(E), 100 * mean(p_paired < p_unpaired)))
cat(sprintf("limma moderated t      : FDR < 0.05 for %d genes | %d of them also significant by the ordinary test\n",
            sum(tl$adj.P.Val < 0.05), sum(tl$adj.P.Val < 0.05 & q_paired < 0.05)))
cat(sprintf("%s under limma: logFC %.4f, moderated t %.3f (ordinary %.3f), p %.3e\n",
            g, tl[g, "logFC"], tl[g, "t"], t_hand, tl[g, "P.Value"]))
cat(sprintf("median |logFC| among the FDR-significant genes: %.3f\n",
            median(abs(tl$logFC[tl$adj.P.Val < 0.05]))))
cat(sprintf("effect-size filter on the limma list: |logFC| > 0.5 -> %d genes | > 1 -> %d genes\n",
            sum(tl$adj.P.Val < 0.05 & abs(tl$logFC) > 0.5), sum(tl$adj.P.Val < 0.05 & abs(tl$logFC) > 1)))
knitr::kable(head(tl[order(-tl$logFC), c("logFC", "t", "P.Value", "adj.P.Val")], 8), digits = 4,
             caption = "Strongest increases in diseased tissue")



Four numbers deserve comment. **3,771 of 5,000 genes** reach FDR < 0.05 with limma and 3,766 with the ordinary paired
t-test, 3,753 of them shared; the empirical-Bayes moderation, which shrinks each gene's residual variance towards a
prior fitted across all genes, changes individual statistics (TNFRSF17 moves from t = 10.252 to 10.848) but barely
changes the count, because with 68 degrees of freedom per gene there is little variance left to borrow. Moderation
earns its keep in the three-versus-three experiments the group runs more often than in this 69-pair cohort. The
unpaired analysis finds 3,420, so the pairing buys about 350 extra discoveries and lowers the p-value for 92 % of
genes, which is what a within-subject design is for.

But a list of 3,771 "significant" genes out of 5,000 is not a result — it says that diseased and healthy gingiva
differ almost everywhere, which is biologically true for an inflamed tissue invaded by plasma cells and useless as a
shortlist. The median significant gene changes by 0.428 log2 units, a factor of 1.35. Adding an effect-size
requirement is therefore not cosmetic: at |logFC| > 1 (a two-fold change) the list falls to **204 genes**, which a
wet-lab collaborator can act on, and the top of that list is dominated by immunoglobulin genes (IGHD, IGHM, IGLL5,
IGLJ3, IGK) — the plasma-cell infiltrate made quantitative.

Note also what the 5,000 genes are: the most *variable* genes of the original 54,675 probe sets, selected when the
course data set was frozen. That pre-filter was applied blind to the diseased/healthy labels, so it does not inflate
the comparison — but a filter chosen with the labels in view (for example "keep genes that differ between groups")
would be the first form of double dipping, and section 8 returns to it.



In [ ]:
pdf1 <- data.table(p_paired = p_paired, p_unpaired = p_unpaired)
pd1 <- ggplot(melt(pdf1, measure.vars = c("p_paired", "p_unpaired")), aes(value, colour = variable)) +
  geom_freqpoly(bins = 50, linewidth = 0.6) +
  scale_colour_manual(values = c(p_paired = PAL[["blue"]], p_unpaired = PAL[["orange"]]),
                      labels = c("paired", "unpaired"), name = NULL) +
  labs(title = "p-value histograms", x = "p-value", y = "genes") +
  theme(legend.position = c(0.7, 0.8), legend.text = element_text(size = 6))
k <- seq_len(nrow(E))
pd2 <- ggplot(data.table(k = k, p = sort(p_paired)), aes(k, p)) + geom_line(colour = PAL[["blue"]]) +
  geom_line(aes(y = 0.05 * k / nrow(E)), linetype = "dashed", colour = "grey35") +
  geom_vline(xintercept = sum(q_paired < 0.05), linetype = "dotted", colour = PAL[["orange"]]) +
  scale_x_log10() + scale_y_log10() +
  labs(title = "Benjamini-Hochberg, drawn", x = "rank k", y = "sorted p-value (dashed: 0.05 k/m)")
pd3 <- ggplot(data.table(u = -log10(p_unpaired), p = -log10(p_paired)), aes(u, p)) +
  geom_point(size = 0.3, alpha = 0.2, colour = PAL[["blue"]]) + geom_abline(linetype = "dashed") +
  labs(title = "Pairing helps almost everywhere",
       x = "-log10 p unpaired", y = "-log10 p paired")
print(pd1 | pd2 | pd3)



<p class="caption"><b>Figure 5.</b> Left: both p-value histograms pile up against zero — the signature of a real and
widespread effect, as opposed to the flat histogram expected under the null. Centre: the Benjamini-Hochberg
construction drawn explicitly; the procedure keeps every gene whose sorted p-value lies below the line 0.05k/m, and
the dotted vertical line marks the 3,766th rank where the curve last crosses it. Right: gene by gene, the paired
analysis gives the smaller p-value in 92 % of cases, with the largest gains for genes whose between-patient
variability is high.</p>

## 5. Volcano, MA and heatmap, done properly

Three plots carry a differential-expression analysis into a paper, and each has a standard way of being misleading.
The **volcano plot** (effect size against significance) invites reading significance as importance; drawing both
thresholds on it, and annotating only genes that pass both, fixes that. The **MA plot** (effect against average
abundance) exists to reveal intensity-dependent bias — a trend in the cloud means the normalisation failed. The
**heatmap** is the most dangerous of the three: selecting the most significant genes and then showing that they
separate the groups is circular, and the picture will look convincing even when built on noise. The honest heatmap
either uses genes chosen independently of the comparison, or is labelled as an illustration of the selection rather
than evidence for it.



In [ ]:
vdt <- data.table(gene = rownames(E), logFC = tl$logFC, p = tl$P.Value, q = tl$adj.P.Val,
                  A = rowMeans(E))
vdt[, sig := q < 0.05 & abs(logFC) > 1]
pv <- ggplot(vdt, aes(logFC, -log10(p), colour = sig)) + geom_point(size = 0.4, alpha = 0.35) +
  geom_vline(xintercept = c(-1, 1), linetype = "dotted") +
  geom_hline(yintercept = -log10(max(vdt$p[vdt$q < 0.05])), linetype = "dotted") +
  ggrepel::geom_text_repel(data = head(vdt[sig == TRUE][order(p)], 6), aes(label = gene),
                           size = 2.2, max.overlaps = 20, colour = "black") +
  scale_colour_manual(values = c(`FALSE` = "grey70", `TRUE` = PAL[["orange"]]), guide = "none") +
  labs(title = sprintf("Volcano: %d genes pass both thresholds", sum(vdt$sig)),
       x = "log2 fold change (diseased - healthy)", y = "-log10 p (paired, moderated)")
pm <- ggplot(vdt, aes(A, logFC, colour = sig)) + geom_point(size = 0.4, alpha = 0.35) +
  geom_hline(yintercept = 0) +
  scale_colour_manual(values = c(`FALSE` = "grey70", `TRUE` = PAL[["orange"]]), guide = "none") +
  labs(title = "MA plot: no intensity-dependent trend",
       x = "average log2 intensity A", y = "M = log2 fold change")
print(pv | pm)



<p class="caption"><b>Figure 6.</b> Left: volcano plot with both thresholds drawn; 204 genes pass FDR < 0.05 and
|logFC| > 1, and the six most significant of them are labelled — all immunoglobulin or plasma-cell genes. Right: the
MA plot shows no trend of the fold change with average intensity, confirming that the RMA normalisation needs no
further correction; the orange points are spread over the whole intensity range rather than concentrated at low
abundance, where measurement noise would create spurious fold changes.</p>



In [ ]:
ord30 <- rownames(tl)[order(-abs(tl$logFC))]
top30 <- head(ord30[tl$adj.P.Val[order(-abs(tl$logFC))] < 0.05], 30)
pc30  <- as.vector(rbind(paste0(patients[1:30], "_D"), paste0(patients[1:30], "_H")))
zmat  <- t(scale(t(E[top30, pc30])))
ann   <- data.frame(site = ifelse(grepl("_D$", pc30), "diseased", "healthy"), row.names = pc30)
pheatmap(zmat, cluster_cols = FALSE, cluster_rows = TRUE, annotation_col = ann,
         annotation_colors = list(site = c(diseased = PAL[["orange"]], healthy = PAL[["blue"]])),
         show_colnames = FALSE, fontsize_row = 5, fontsize = 7, breaks = seq(-2.5, 2.5, length.out = 101),
         main = "Top 30 genes by |logFC| - selected on these same samples (see the warning below)")



<p class="caption"><b>Figure 7.</b> Heatmap of the thirty largest fold changes across the first thirty patients, each
pair shown as diseased then healthy, rows standardised. The alternating pattern is striking — and partly guaranteed,
because the rows were chosen for having a large difference in these very samples. The honest caption is the one
written here: this figure illustrates the selected genes, it does not provide independent evidence that they
differ.</p>

## 6. Over-representation analysis, by hand

The gene list must now be turned into biology. The simplest method asks, for each gene set, whether the overlap with
our list is larger than chance under a hypergeometric distribution: drawing \(n\) genes (the list) without replacement
from \(N\) genes (the universe) of which \(K\) belong to the set, the probability of seeing at least \(k\) in common is

\[ P(X \ge k) = \sum_{j=k}^{\min(n,K)} \frac{\binom{K}{j}\binom{N-K}{n-j}}{\binom{N}{n}}, \]

which base R evaluates as `phyper(k - 1, K, N - K, n, lower.tail = FALSE)`. Two decisions matter more than the formula.
The **universe** \(N\) must be the genes that could have been detected — here the 5,000 genes in the matrix, not the
20,000 genes of the genome, or every p-value is wrong in the optimistic direction. And the **list** depends on the
thresholds of section 4, so an over-representation result inherits every arbitrary choice made there.



In [ ]:
read_gmt <- function(path, universe, min_size = 10, max_size = 500) {
  ln  <- strsplit(readLines(path), "\t")
  out <- lapply(ln, function(f) intersect(f[-(1:2)], universe))
  names(out) <- vapply(ln, `[`, "", 1)
  out[lengths(out) >= min_size & lengths(out) <= max_size]
}
universe <- rownames(E)
kegg <- read_gmt(dp("genesets_kegg_2021.gmt"), universe)
gobp <- read_gmt(dp("genesets_gobp_2023.gmt"), universe)
up   <- rownames(E)[tl$adj.P.Val < 0.05 & tl$logFC >  0.5]
down <- rownames(E)[tl$adj.P.Val < 0.05 & tl$logFC < -0.5]
ora <- function(selected, sets, N) {
  r <- rbindlist(lapply(names(sets), function(nm) {
    m <- sets[[nm]]; k <- length(intersect(selected, m))
    if (k == 0) return(NULL)
    data.table(set = nm, size = length(m), overlap = k, expected = length(selected) * length(m) / N,
               p = phyper(k - 1, length(m), N - length(m), length(selected), lower.tail = FALSE))
  }))
  r <- r[order(p)]; r[, q := p.adjust(p, "BH")][]
}
ora_up <- ora(up, kegg, length(universe)); ora_dn <- ora(down, kegg, length(universe))
cat(sprintf("gene sets kept (10-500 members inside the universe): KEGG %d, GO-BP %d\n", length(kegg), length(gobp)))
cat(sprintf("list sizes: %d up-regulated, %d down-regulated, universe %d\n", length(up), length(down), length(universe)))
cat(sprintf("KEGG sets at FDR < 0.05: %d for the up-regulated list, %d for the down-regulated one\n",
            sum(ora_up$q < 0.05), sum(ora_dn$q < 0.05)))
knitr::kable(head(ora_up, 8), digits = c(0, 0, 0, 2, 20, 20))


In [ ]:
r1 <- ora_up[1]; N <- length(universe)
cat(sprintf("worked by hand for '%s':\n", r1$set))
cat(sprintf("   universe N = %d, set size K = %d, list size n = %d, overlap k = %d\n",
            N, r1$size, length(up), r1$overlap))
cat(sprintf("   expected overlap by chance = n*K/N = %.2f, observed = %d, fold enrichment = %.2f\n",
            r1$expected, r1$overlap, r1$overlap / r1$expected))
cat(sprintf("   P(X >= k) = %.3e   [phyper(%d, %d, %d, %d, lower.tail = FALSE)]\n",
            phyper(r1$overlap - 1, r1$size, N - r1$size, length(up), lower.tail = FALSE),
            r1$overlap - 1, r1$size, N - r1$size, length(up)))
ora_up_go <- ora(up, gobp, N)
cat(sprintf("\nGO biological processes at FDR < 0.05 for the up list: %d of %d tested\n",
            sum(ora_up_go$q < 0.05), nrow(ora_up_go)))
knitr::kable(head(ora_up_go, 6), digits = c(0, 0, 0, 2, 20, 20))



The output is biologically coherent, which is the first thing to check: neutrophil extracellular trap formation
(24 of the 37 genes of the pathway in a list of 739, against 5.47 expected — a 4.4-fold enrichment at p = 4.4e-12),
cytokine-cytokine receptor interaction, leukocyte transendothelial migration and B-cell activation are exactly what an
inflamed, plasma-cell-rich gingival lesion should show<sup class="c"><a href="#r14">[14]</a></sup>. Note two structural
features of the output. Forty-four KEGG pathways are significant for the up-regulated list and **none** for the
down-regulated one, and the GO-BP results (163 significant terms of 1,270 tested) contain large families of
overlapping terms — "B cell activation", "B cell receptor signaling pathway" and "antigen receptor-mediated
signaling" share most of their genes. GO terms are a hierarchy, so their tests are far from independent and the
significant list is highly redundant; reporting the top twenty terms is reporting one finding twenty times.
Summarising by semantic clustering, or showing only the most specific term of each branch, is the standard remedy.

## 7. Gene-set enrichment analysis with fgsea

Over-representation throws away two things: the ranking inside the list, and every gene that missed the threshold. A
pathway in which a hundred genes all move slightly upwards may contain no gene at FDR < 0.05 and still be the most
interesting result in the study. Gene-set enrichment analysis
(GSEA)<sup class="c"><a href="#r6">[6]</a></sup><sup class="c"><a href="#r7">[7]</a></sup> was designed for that case.
Rank all genes by a signed statistic, walk down the ranking, add to a running sum when a gene belongs to the set
(weighted by the magnitude of its statistic) and subtract when it does not; the **enrichment score** is the maximum
deviation of that walk from zero. Significance comes from a permutation null, and the score is normalised by the mean
of the null values of the same sign to give the **normalised enrichment score (NES)**, which can be compared across
sets of different size.

`fgsea`<sup class="c"><a href="#r30">[30]</a></sup> implements this with a cumulative-statistic algorithm that makes
the permutation step orders of magnitude cheaper than the naive loop, and it estimates p-values from a fitted
distribution of the sampled null rather than from the raw count, so a p-value is not floored at 1/(n+1) — the problem
Phipson and Smyth analyse<sup class="c"><a href="#r3">[3]</a></sup>. We force the serial backend so that the
permutation stream is reproducible from the seed.

Two permutation schemes exist and they are not equivalent. Permuting **sample labels** rebuilds the whole statistic
and preserves the correlation between genes. Permuting **gene labels** — what `fgsea` does — is far cheaper but treats
genes as exchangeable, which they are not, and is known to be anti-conservative for correlated
sets<sup class="c"><a href="#r23">[23]</a></sup><sup class="c"><a href="#r26">[26]</a></sup>. Using it and stating the
consequence is the honest compromise; `limma::camera` is the alternative that corrects for inter-gene correlation.



In [ ]:
stats_rank <- setNames(tl$t, rownames(E))             # ranked by the moderated t statistic
set.seed(0)
fg  <- fgsea(kegg, stats_rank, minSize = 10, maxSize = 500, BPPARAM = BiocParallel::SerialParam())
set.seed(0)
fgo <- fgsea(gobp, stats_rank, minSize = 10, maxSize = 500, BPPARAM = BiocParallel::SerialParam())
cat(sprintf("fgsea on KEGG : %d sets at padj < 0.05 (%d with NES > 0, %d with NES < 0)\n",
            sum(fg$padj < 0.05), sum(fg$padj < 0.05 & fg$NES > 0), sum(fg$padj < 0.05 & fg$NES < 0)))
cat(sprintf("fgsea on GO-BP: %d sets at padj < 0.05 (%d positive)\n",
            sum(fgo$padj < 0.05), sum(fgo$padj < 0.05 & fgo$NES > 0)))
knitr::kable(head(as.data.frame(fg[order(pval), .(pathway, size, ES, NES, pval, padj)]), 8), digits = 3)
sets_ora  <- ora_up[q < 0.05, set]
sets_gsea <- fg[padj < 0.05 & NES > 0, pathway]
cat(sprintf("\nORA-significant %d | fgsea-positive %d | shared %d | ORA only %d | fgsea only %d\n",
            length(sets_ora), length(sets_gsea), length(intersect(sets_ora, sets_gsea)),
            length(setdiff(sets_ora, sets_gsea)), length(setdiff(sets_gsea, sets_ora))))
cat("found only by fgsea:", paste(head(sort(setdiff(sets_gsea, sets_ora)), 5), collapse = " | "), "\n")
cat("found only by ORA  :", paste(sort(setdiff(sets_ora, sets_gsea)), collapse = " | "), "\n")


In [ ]:
best <- fg[order(pval)][NES > 0][1]$pathway
pe <- plotEnrichment(kegg[[best]], stats_rank) +
  labs(title = sprintf("%s (ES = %.3f, NES = %.2f)", substr(best, 1, 38),
                       fg[pathway == best, ES], fg[pathway == best, NES]),
       x = "rank in the list (most up-regulated first)", y = "running enrichment score")
cmp <- merge(fg[, .(set = pathway, NES, padj_gsea = padj)], ora_up[, .(set, p_ora = p)], by = "set")
pc2 <- ggplot(cmp, aes(-log10(p_ora), NES, colour = padj_gsea < 0.05)) + geom_point(size = 1.1) +
  geom_hline(yintercept = 0) +
  scale_colour_manual(values = c(`FALSE` = "grey70", `TRUE` = PAL[["orange"]]),
                      name = "fgsea padj < 0.05") +
  labs(title = "The two methods broadly agree", x = "-log10 p (ORA, up list)", y = "fgsea NES") +
  theme(legend.position = "bottom", legend.text = element_text(size = 6))
print(pe | pc2)



<p class="caption"><b>Figure 8.</b> Left: the running enrichment score for cytokine-cytokine receptor interaction, the
most significant positively enriched KEGG pathway (ES 0.526, NES 2.59 over 104 genes); the score climbs steeply because
its genes (ticks) are concentrated at the top of the ranking, and the maximum is reached about a fifth of the way down.
Right: fgsea NES against the over-representation p-value, set by set — the two agree in direction and broadly in
strength, and 43 of the 44 pathways found by over-representation are also found by fgsea, which additionally reports
26 that the threshold-based method missed, including apoptosis, antigen processing and presentation and allograft
rejection.</p>

The reconciliation rule is simple. Sets found by both are solid. Sets found only by GSEA are typically broad and
moderately shifted — plausible but dependent on the permutation scheme, and with gene-label permutation their
p-values are optimistic. Sets found only by over-representation (here exactly one, Legionellosis) are usually driven
by a handful of very large fold changes, so they survive a threshold but not a whole-distribution test. Report the
method, the gene-set database with its version, the universe, the permutation scheme and the number of permutations;
without those five items an enrichment table cannot be
reproduced<sup class="c"><a href="#r21">[21]</a></sup><sup class="c"><a href="#r22">[22]</a></sup>.

<div class="exercise"><h4>Exercise 2</h4>
<p>Rerun the <code>fgsea</code> call ranking genes by <code>tl$logFC</code> instead of by the moderated t statistic.
Which pathways change rank, and why does ranking by effect size alone promote noisy, low-expression genes? Then run
<code>limma::camera(E, kegg, design, contrast = "status")</code> on the same gene sets and compare its ranking with
fgsea's: camera corrects for inter-gene correlation, so the sets whose significance collapses are precisely the
correlated ones the gene-label permutation flattered.</p></div>

## 8. The four ways an omics analysis goes wrong

<b>p much greater than n.</b> Five thousand tests on 69 pairs: even under a perfect null, 250 genes would reach
p &lt; 0.05 by chance. The remedy — false-discovery-rate
control<sup class="c"><a href="#r11">[11]</a></sup><sup class="c"><a href="#r8">[8]</a></sup> — is standard, and it
remains valid under the positive dependence that correlated genes
produce<sup class="c"><a href="#r29">[29]</a></sup>, but its validity depends on the p-values being correctly
calibrated, which brings the next point.

<b>The permutation scheme must respect the design.</b> Shuffling the diseased/healthy labels across all 138 samples
breaks the pairing and treats two samples from one patient as independent; flipping the two labels *within* each
patient is the correct paired permutation. The chunk below runs both, once and then twenty times, because a single
draw turns out to be a very noisy way to make this point.

<b>Double dipping.</b> Choosing genes with the labels in view and then evaluating a classifier — or a heatmap, or a
test — on the same samples is circular<sup class="c"><a href="#r12">[12]</a></sup>. The demonstration below uses
*random* labels, where the true accuracy is 0.5 by construction.

<b>The winner's curse.</b> The effect sizes of the genes selected for being the most extreme are biased upwards, so
they shrink in an independent cohort<sup class="c"><a href="#r24">[24]</a></sup><sup class="c"><a href="#r25">[25]</a></sup>.
Splitting the patients in two halves measures that shrinkage directly.



In [ ]:
count_sig <- function(p) c(raw = sum(p < 0.05), fdr = sum(p.adjust(p, "BH") < 0.05))
perm_across <- function(seed) {
  set.seed(seed); yw <- sample(status)
  count_sig(apply(E, 1, function(v) t.test(v[yw == 1], v[yw == 0], var.equal = TRUE)$p.value))
}
perm_within <- function(seed) {
  set.seed(seed); fl <- sample(c(TRUE, FALSE), length(patients), TRUE)
  Dw <- Dm; Hw <- Hm; Dw[, fl] <- Hm[, fl]; Hw[, fl] <- Dm[, fl]
  d <- Dw - Hw; tp <- rowMeans(d) / (apply(d, 1, sd) / sqrt(ncol(d)))
  count_sig(2 * pt(-abs(tp), ncol(d) - 1))
}
acr <- sapply(1:20, perm_across); wit <- sapply(1:20, perm_within)
cat(sprintf("one draw, ACROSS patients: raw p < 0.05 for %d genes, FDR < 0.05 for %d\n", acr[1, 1], acr[2, 1]))
cat(sprintf("one draw, WITHIN patients: raw p < 0.05 for %d genes, FDR < 0.05 for %d\n", wit[1, 2], wit[2, 2]))
cat(sprintf("20 draws, ACROSS patients: raw mean %.0f [%d, %d], FDR mean %.1f\n",
            mean(acr[1, ]), min(acr[1, ]), max(acr[1, ]), mean(acr[2, ])))
cat(sprintf("20 draws, WITHIN patients: raw mean %.0f [%d, %d], FDR mean %.1f\n",
            mean(wit[1, ]), min(wit[1, ]), max(wit[1, ]), mean(wit[2, ])))
cat(sprintf("expected under a correct null: about %d genes at raw p < 0.05\n", round(0.05 * nrow(E))))



This experiment is the one place where the R edition disagrees with the Python edition, and the disagreement is the
finding. The Python notebook ran one permutation of each kind and obtained 1,256 genes at raw p < 0.05 from the
across-patient scheme against 69 from the within-patient flip, and read that as the wrong scheme inflating false
positives fivefold. Our own single draws point the same way (458 against 130) — and twenty repetitions of each scheme
show that neither pair of numbers supports the reading. The mean count is 194 across patients and 250 within, both
compatible with the nominal 250, and the counts range from 36 to 996 and from 16 to 733 respectively: with 5,000
strongly correlated genes, the number of nominally significant tests in any one permutation is itself a highly
variable quantity, and a single draw is simply not an estimate. What does survive the repetition is that
Benjamini-Hochberg control holds under both schemes — no FDR-significant gene in any of the forty draws, against 3,771
in the real comparison. The methodological rule stands, but it stands on its premise rather than on this count: two
samples from one patient are correlated, so a permutation that mixes them does not generate a valid null for a test
that assumes independence. Permute within the pairing because the design says so, and do not expect a tidy count to
prove it for you.



In [ ]:
set.seed(3)
fl <- sample(c(TRUE, FALSE), length(patients), TRUE)
y_null <- as.integer(unlist(lapply(fl, function(f) if (f) c(1, 0) else c(0, 1))))   # random labels, 1 per pair
pair_cols <- as.vector(rbind(paste0(patients, "_D"), paste0(patients, "_H")))
Xp <- t(E[, pair_cols]); gp <- rep(patients, each = 2)
fsel <- function(Xtr, ytr, k = 50) {                   # simple two-sample filter, the analogue of SelectKBest
  sc <- apply(Xtr, 2, function(v) {
    m <- tapply(v, ytr, mean); va <- tapply(v, ytr, var); abs(diff(m)) / sqrt(sum(va))
  })
  order(-sc)[1:k]
}
folds_p <- group_folds(gp)                             # whole patients stay in one fold
cv_logit <- function(fixed_sel) mean(sapply(1:5, function(f) {
  tr <- folds_p != f
  s  <- if (is.null(fixed_sel)) fsel(Xp[tr, ], y_null[tr]) else fixed_sel
  m  <- suppressWarnings(glm(y ~ ., data = data.frame(Xp[tr, s], y = y_null[tr]), family = binomial))
  mean(as.integer(predict(m, data.frame(Xp[!tr, s]), type = "response") > 0.5) == y_null[!tr])
}))
cat("RANDOM labels (true accuracy 0.500):\n")
cat(sprintf("   gene selection outside the cross-validation : %.3f\n", cv_logit(fsel(Xp, y_null))))
cat(sprintf("   gene selection inside each training fold    : %.3f\n", cv_logit(NULL)))


In [ ]:
set.seed(0); perm <- sample(length(patients))
h1 <- patients[perm[1:34]]; h2 <- patients[perm[35:length(patients)]]
paired_stats <- function(ps) {
  d <- E[, paste0(ps, "_D")] - E[, paste0(ps, "_H")]
  t <- rowMeans(d) / (apply(d, 1, sd) / sqrt(length(ps)))
  list(fc = rowMeans(d), t = t, p = 2 * pt(-abs(t), length(ps) - 1))
}
s1 <- paired_stats(h1); s2 <- paired_stats(h2)
sel <- order(-abs(s1$t))[1:100]
cat("top 100 genes by |t| in the discovery half (34 patients):\n")
cat(sprintf("   mean |log2FC| in the discovery half  : %.3f\n", mean(abs(s1$fc[sel]))))
cat(sprintf("   mean |log2FC| in the validation half : %.3f (shrinkage %.0f %%)\n",
            mean(abs(s2$fc[sel])), 100 * (1 - mean(abs(s2$fc[sel])) / mean(abs(s1$fc[sel])))))
cat(sprintf("   replicating at p < 0.05 with the same sign: %d of 100\n",
            sum(s2$p[sel] < 0.05 & sign(s1$fc[sel]) == sign(s2$fc[sel]))))


In [ ]:
nulldf <- rbind(data.table(scheme = "across patients", n = acr[1, ]),
                data.table(scheme = "within patients", n = wit[1, ]))
pp1 <- ggplot(nulldf, aes(n, scheme, colour = scheme)) +
  geom_point(size = 1.4, alpha = 0.7, position = position_jitter(height = 0.12, seed = 1)) +
  geom_vline(xintercept = 0.05 * nrow(E), linetype = "dotted") +
  scale_x_log10() + scale_colour_manual(values = unname(PAL[c("orange", "blue")]), guide = "none") +
  labs(title = "Genes at raw p < 0.05 in 20 null draws", x = "genes (log scale; dotted = 250 expected)", y = NULL)
pp2 <- ggplot(data.table(what = c("selection\noutside CV", "selection\ninside CV"),
                         acc = c(cv_logit(fsel(Xp, y_null)), cv_logit(NULL))),
              aes(what, acc, fill = what)) + geom_col(width = 0.6) +
  geom_hline(yintercept = 0.5, linetype = "dotted") +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")]), guide = "none") +
  labs(title = "Double dipping, measured", x = NULL, y = "accuracy on RANDOM labels")
pp3 <- ggplot(data.table(d = abs(s1$fc[sel]), v = abs(s2$fc[sel])), aes(d, v)) +
  geom_point(size = 1.1, colour = PAL[["blue"]]) + geom_abline(linetype = "dashed") +
  labs(title = "The winner's curse", x = "|log2FC| in the discovery half", y = "|log2FC| in the validation half")
print(pp1 | pp2 | pp3)



<p class="caption"><b>Figure 9.</b> Left: the number of genes at raw p &lt; 0.05 in twenty null draws of each
permutation scheme (logarithmic axis, dotted line at the 250 expected). Both schemes scatter over more than an order of
magnitude — 36 to 996 genes across patients, 16 to 733 within — and neither cloud is centred away from the nominal
value, so this statistic does not discriminate between the two schemes. Centre: with labels assigned at
random, selecting the fifty best genes before cross-validation gives 0.610 accuracy where the truth is 0.500, while
selecting inside each fold gives 0.485 — noise around chance. Right: the hundred genes with the largest |t| in a random
half of the patients lose 9 % of their apparent effect in the other half, with almost every point below the
diagonal.</p>

The winner's-curse panel also carries good news that should be stated, since this unit is otherwise a catalogue of
traps: all 100 selected genes replicate at p &lt; 0.05 with the same sign in the validation half. When an effect is
large, widespread and measured in a paired design, omics results do replicate — the 9 % shrinkage is the honest
correction to apply to the headline effect sizes, not a reason to distrust the finding. What the single cohort cannot
deliver is generalisation: one population, one clinical protocol, one array platform, and — as the series metadata make
explicit — no age, sex or smoking information with which to check that the comparison is not confounded by something
the sampling happened to align with.

## 9. Summary

Omics analysis is ordinary statistics applied under three unusual conditions: thousands of variables, few and
non-independent samples, and a result that is a list rather than a prediction. The sequence half of this unit made the
non-independence visible in the 621 reviewed UniProt sequences: a nearest-neighbour family classifier on dipeptide
profiles scored 0.937 under a random split, 0.905 once homologous sequences were kept in one fold and 0.879 when
grouped by protein name, while multinomial logistic regression on the same features lost only 0.990 to 0.981 — and the
diagnostic behind those numbers is that the random split leaves 21.9 % of test sequences a relative above 0.9
similarity in the training set, against 0 % above 0.7 after grouping. The rule that follows is to cluster first
(345 homology groups here, 256 of them singletons) and let the clusters be the groups of the cross-validator, reporting
the grouped number as the performance of the method; the gap grows from 0.071 to 0.120 as the training set shrinks from
240 sequences to 30.

The expression half used the frozen gingival-tissue matrix: 5,000 genes, 138 samples, 69 within-patient pairs, log2 RMA
intensities with a library-size proxy varying by 2.2 % and no demographic covariates at all. PC1 carried 27 % of the
variance and separated diseased from healthy sites (p = 4.3e-22); PC2 carried 13 % and did not (p = 0.093). A paired
linear model with one intercept per patient reproduced the paired t-test exactly (TNFRSF17: beta = 2.1312, t = 10.252,
p = 2.0e-15, against t = 8.250 unpaired), limma's empirical-Bayes moderation moved that gene to t = 10.848, and over the
whole matrix the pairing gave the smaller p-value for 92 % of genes and 3,771 discoveries at FDR &lt; 0.05 against 3,420
unpaired. That number is also a warning: with three-quarters of the measured genes significant, statistical
significance has stopped being a filter, and the effect-size criterion |logFC| &gt; 1 — which leaves 204 genes, led by
the immunoglobulins — is what makes the list actionable.

Over-representation analysis written from the hypergeometric distribution found 44 KEGG pathways for the up-regulated
list and none for the down-regulated one, led by neutrophil extracellular trap formation (24 of 37 genes against 5.47
expected, p = 4.4e-12); `fgsea` on the moderated t statistics found 69 positively enriched KEGG sets, 43 of them shared
with over-representation, and the two were reconciled set by set rather than reported separately. The closing
experiments quantified the classic failures: selecting genes before cross-validation rather than inside it raised the
accuracy on random labels from 0.485 to 0.610; the hundred strongest genes of a discovery half lost 9 % of their effect
size in the validation half, although all hundred replicated in sign and significance; and twenty repetitions of each
permutation scheme showed that the count of nominally significant genes under the null varies over more than an order
of magnitude whichever scheme is used (36-996 across patients, 16-733 within, both centred near the nominal 250), so
the single-draw contrast the Python edition reads as fivefold inflation is not evidence — the case for permuting within
the pairing rests on the design, and FDR control held in all forty draws. Three habits are worth carrying into the group's periodontitis work: write
down the universe, the thresholds and the permutation scheme before computing anything; make the patient, not the
sample, the unit of analysis; and treat every list as provisional until it has been seen in a second
cohort<sup class="c"><a href="#r9">[9]</a></sup><sup class="c"><a href="#r12">[12]</a></sup>.

<div class="box bridge"><h4>Bridge</h4>
<p>Within this course: the FDR machinery is <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a>, the linear model
is <a href="../U9_Linear_Models/index.html">Unit 9</a>, the group splits and leakage are
<a href="../U12_Validation/index.html">Unit 12</a>, PCA and feature selection are
<a href="../U13_Features_Dimensionality/index.html">Unit 13</a>, and the clustering used on expression profiles is
<a href="../U15_Clustering/index.html">Unit 15</a>; the data-quality and consent questions around human samples were
raised in <a href="../U6_Data_Quality/index.html">Unit 6</a>. The ranked-list analytics of
<a href="../U19_Virtual_Screening/index.html">Unit 19</a> are the chemical twin of the enrichment analysis done here,
and <a href="../U20_MD_Analytics/index.html">Unit 20</a> applies the same multiple-comparison discipline to per-residue
simulation data. The Python edition of this unit, which uses Biopython and an OLS implementation of the paired model
instead of <code>seqinr</code> and <code>limma</code>, is <a href="../../Python/U21_Omics/index.html">here</a>.
In the sister courses: sequence handling, alignment and homology are
<a href="../../../BGM324/01_Sequence_Alignment/index.html">BGM324 unit 1</a>, protein
structure is <a href="../../../BGM324/02_Protein_Structure_Prediction/index.html">BGM324 unit 2</a>,
and the machine-learning protocol used for the classifier is in
<a href="../../../BGM411/R/index.html">BGM411 (R edition)</a>.</p></div>

## 10. Self-assessment

<details><summary>1. Why is a random train/test split wrong for a protein-family classifier, and what replaces it?</summary>
<p>Because homologous sequences are near-duplicates: a random split leaves a close relative of almost every test
sequence in the training set, so the model can memorise rather than generalise. Cluster the sequences at an identity
threshold (CD-HIT, MMseqs2 or the pre-computed UniRef clusters) and use those clusters as the groups of a group-aware
cross-validator; the grouping variable can never be the class label itself. Here the nearest-neighbour model fell from
0.937 to 0.905 and 21.9 % of test sequences lost a near-identical relative from the training set.</p></details>

<details><summary>2. These data are log2 RMA microarray intensities. Why not use DESeq2?</summary>
<p>DESeq2 models integer counts with a negative-binomial mean-variance relationship produced by sequencing. Microarray
intensities are continuous and already normalised, so the count model does not apply. The right tool is a per-gene
linear model with moderated variance — limma — which is what the paired model here implements.</p></details>

<details><summary>3. What does the patient term in the linear model do, and what is the price of including it?</summary>
<p>It absorbs every stable between-patient difference, so the status coefficient is estimated from within-patient
contrasts only; this is why 92 % of genes get a smaller p-value. The price is degrees of freedom — 68 instead of 136 —
and the inability to estimate any effect that does not vary within a patient, such as a patient-level covariate.</p></details>

<details><summary>4. What does limma's empirical-Bayes moderation add, and why did it change so little here?</summary>
<p>It shrinks each gene's residual variance towards a prior fitted across all genes, which stabilises the t statistic
of genes whose variance was underestimated by chance. With 68 residual degrees of freedom per gene the ordinary
variance is already well estimated, so the discovery count moved only from 3,766 to 3,771. In a three-versus-three
experiment, where each gene has 4 degrees of freedom, the same moderation is the difference between a usable and an
unusable analysis.</p></details>

<details><summary>5. 3,771 of 5,000 genes are significant. Is that a good result?</summary>
<p>It is a correct result and a useless list. With a tissue-wide inflammatory response the null is false nearly
everywhere, so significance no longer discriminates. Add an effect-size threshold (204 genes at |logFC| &gt; 1), report
the median effect size of the significant set (0.428 log2 units here), and move the interpretation to gene sets.</p></details>

<details><summary>6. Why must the universe of an over-representation test be the measured genes rather than the genome?</summary>
<p>Because the hypergeometric model asks how surprising the overlap is given what could have been drawn. Using 20,000
genes when only 5,000 could be selected inflates the apparent enrichment and shrinks every p-value. The universe, the
list and the thresholds must all be reported.</p></details>

<details><summary>7. What does GSEA see that over-representation cannot?</summary>
<p>Coordinated, moderate shifts: a set whose genes all move slightly in the same direction without any single gene
passing the threshold. GSEA uses the whole ranking and the magnitude of each statistic, so it needs no threshold at
all. Here it recovered 43 of the 44 over-representation hits plus 26 further sets.</p></details>

<details><summary>8. Gene-label and sample-label permutation give different GSEA p-values. Which is right?</summary>
<p>Sample-label permutation, because it preserves the correlation structure between genes; gene-label permutation —
what fgsea does — treats genes as exchangeable and is anti-conservative for correlated sets. It is acceptable when the
sample size is too small to permute, provided the choice is stated; <code>limma::camera</code> is the competitive test
that corrects for inter-gene correlation.</p></details>

<details><summary>9. You permute the diseased/healthy labels of all 138 samples and find 458 genes at raw p &lt; 0.05 where 250 are expected. Is that proof that the permutation scheme is wrong?</summary>
<p>No — it is one draw from a very wide distribution. Twenty draws of the across-patient scheme gave 36 to 996 genes
(mean 194) and twenty draws of the correct within-patient flip gave 16 to 733 (mean 250), so this count does not
distinguish the two schemes at all. The scheme is still wrong in principle, because samples from one patient are
correlated and the unpaired test assumes they are not; that argument comes from the design, not from the count. FDR
control held under both schemes in all forty draws.</p></details>

<details><summary>10. A colleague selects the 50 most differential genes, trains a classifier with cross-validation and reports 92 % accuracy. What is the flaw and how is it fixed?</summary>
<p>The selection used the labels of all samples, including those in each test fold, so information leaked. Put the
selection inside the cross-validation loop so it is refitted on each training fold. On random labels, that distinction
moved the accuracy from 0.610 to 0.485 in this data set.</p></details>

<details><summary>11. What is the winner's curse and how large was it here?</summary>
<p>Genes selected for having the most extreme statistics have effect sizes biased upwards by the selection, so they
shrink in independent data. The hundred strongest genes of a discovery half of the patients lost 9 % of their mean
|log2FC| in the validation half — while all hundred still replicated in sign and significance.</p></details>

<details><summary>12. Which five items must accompany an enrichment table for it to be reproducible?</summary>
<p>The method (ORA or GSEA, and which statistic was ranked), the gene-set database with its version and date, the
universe of genes, the selection thresholds or the ranking metric, and the permutation scheme with the number of
permutations. Multiple-testing correction and the software versions complete the list.</p></details>

## Glossary

<dl class="gloss">
<dt>FASTA</dt><dd>Plain-text format for sequences: a header line starting with "&gt;" followed by the residues; read here with <code>seqinr::read.fasta</code>.</dd>
<dt>k-mer features</dt><dd>Counts or frequencies of all substrings of length k (here dipeptides, 400 of them) used as a fixed-length numeric representation of a sequence.</dd>
<dt>Group split</dt><dd>Cross-validation in which all members of a group (homology cluster, patient, scaffold) stay in the same fold.</dd>
<dt>Orthologue / paralogue</dt><dd>The same protein in two species / two copies within one genome; both are homologues and both make two rows of a sequence table statistically dependent.</dd>
<dt>log2 RMA intensity</dt><dd>Robust multi-array average summary of a microarray probe set on a log2 scale; a difference of 1 is a two-fold change.</dd>
<dt>Paired design</dt><dd>Two measurements from the same unit (here the diseased and healthy site of one patient), analysed through their difference.</dd>
<dt>log2 fold change</dt><dd>Mean within-patient difference on the log2 scale; the effect size of a differential-expression analysis.</dd>
<dt>Empirical-Bayes moderation</dt><dd>limma's shrinkage of each gene's residual variance towards a prior fitted across all genes, giving the moderated t statistic.</dd>
<dt>Benjamini-Hochberg FDR</dt><dd>Procedure keeping all genes whose sorted p-value is below 0.05k/m, controlling the expected proportion of false discoveries among those declared.</dd>
<dt>Volcano plot</dt><dd>Effect size against -log10 p; both thresholds must be drawn for it to be read correctly.</dd>
<dt>MA plot</dt><dd>Effect size against average intensity; a trend indicates a normalisation problem.</dd>
<dt>Over-representation analysis (ORA)</dt><dd>Hypergeometric test of the overlap between a selected gene list and a gene set, relative to a declared universe.</dd>
<dt>Gene-set enrichment analysis (GSEA)</dt><dd>Threshold-free test using the whole ranking; the enrichment score is the maximum deviation of a weighted running sum.</dd>
<dt>Normalised enrichment score (NES)</dt><dd>Enrichment score divided by the mean of the same-sign permutation scores, comparable across set sizes.</dd>
<dt>Double dipping / circular analysis</dt><dd>Using the same data to select features and to evaluate them, which inflates every subsequent statistic.</dd>
<dt>Winner's curse</dt><dd>Upward bias of the effect sizes of the items selected for being most extreme; measured by replication in held-out samples.</dd>
<dt>Batch effect</dt><dd>Technical variation associated with processing groups; dangerous when confounded with the condition of interest.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Charif D, Lobry JR. SeqinR 1.0-2: a contributed package to the R project for statistical computing devoted to biological sequences retrieval and analysis. In: <em>Structural Approaches to Sequence Evolution</em>. Springer; 2007:207-232. <a href="https://doi.org/10.1007/978-3-540-35306-5_10">doi:10.1007/978-3-540-35306-5_10</a></li>
<li id="r2">Love MI, Huber W, Anders S. Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2. <em>Genome Biol</em>. 2014;15:550. <a href="https://doi.org/10.1186/s13059-014-0550-8">doi:10.1186/s13059-014-0550-8</a></li>
<li id="r3">Phipson B, Smyth GK. Permutation p-values should never be zero: calculating exact p-values when permutations are randomly drawn. <em>Stat Appl Genet Mol Biol</em>. 2010;9:Article39. <a href="https://doi.org/10.2202/1544-6115.1585">doi:10.2202/1544-6115.1585</a></li>
<li id="r4">Ritchie ME, Phipson B, Wu D, et al. limma powers differential expression analyses for RNA-sequencing and microarray studies. <em>Nucleic Acids Res</em>. 2015;43(7):e47. <a href="https://doi.org/10.1093/nar/gkv007">doi:10.1093/nar/gkv007</a></li>
<li id="r5">Smyth GK. Linear models and empirical Bayes methods for assessing differential expression in microarray experiments. <em>Stat Appl Genet Mol Biol</em>. 2004;3:Article3. <a href="https://doi.org/10.2202/1544-6115.1027">doi:10.2202/1544-6115.1027</a></li>
<li id="r6">Subramanian A, Tamayo P, Mootha VK, et al. Gene set enrichment analysis: a knowledge-based approach for interpreting genome-wide expression profiles. <em>Proc Natl Acad Sci USA</em>. 2005;102(43):15545-15550. <a href="https://doi.org/10.1073/pnas.0506580102">doi:10.1073/pnas.0506580102</a></li>
<li id="r7">Mootha VK, Lindgren CM, Eriksson KF, et al. PGC-1alpha-responsive genes involved in oxidative phosphorylation are coordinately downregulated in human diabetes. <em>Nat Genet</em>. 2003;34(3):267-273. <a href="https://doi.org/10.1038/ng1180">doi:10.1038/ng1180</a></li>
<li id="r8">Storey JD, Tibshirani R. Statistical significance for genomewide studies. <em>Proc Natl Acad Sci USA</em>. 2003;100(16):9440-9445. <a href="https://doi.org/10.1073/pnas.1530509100">doi:10.1073/pnas.1530509100</a></li>
<li id="r9">Leek JT, Scharpf RB, Bravo HC, et al. Tackling the widespread and critical impact of batch effects in high-throughput data. <em>Nat Rev Genet</em>. 2010;11(10):733-739. <a href="https://doi.org/10.1038/nrg2825">doi:10.1038/nrg2825</a></li>
<li id="r10">Johnson WE, Li C, Rabinovic A. Adjusting batch effects in microarray expression data using empirical Bayes methods. <em>Biostatistics</em>. 2007;8(1):118-127. <a href="https://doi.org/10.1093/biostatistics/kxj037">doi:10.1093/biostatistics/kxj037</a></li>
<li id="r11">Benjamini Y, Hochberg Y. Controlling the false discovery rate: a practical and powerful approach to multiple testing. <em>J R Stat Soc B</em>. 1995;57(1):289-300. <a href="https://doi.org/10.1111/j.2517-6161.1995.tb02031.x">doi:10.1111/j.2517-6161.1995.tb02031.x</a></li>
<li id="r12">Kriegeskorte N, Simmons WK, Bellgowan PSF, Baker CI. Circular analysis in systems neuroscience: the dangers of double dipping. <em>Nat Neurosci</em>. 2009;12(5):535-540. <a href="https://doi.org/10.1038/nn.2303">doi:10.1038/nn.2303</a></li>
<li id="r13">Barrett T, Wilhite SE, Ledoux P, et al. NCBI GEO: archive for functional genomics data sets - update. <em>Nucleic Acids Res</em>. 2013;41(D1):D991-D995. <a href="https://doi.org/10.1093/nar/gks1193">doi:10.1093/nar/gks1193</a></li>
<li id="r14">Papapanou PN, Behle JH, Kebschull M, et al. Subgingival bacterial colonization profiles correlate with gingival tissue gene expression. <em>BMC Microbiol</em>. 2009;9:221. <a href="https://doi.org/10.1186/1471-2180-9-221">doi:10.1186/1471-2180-9-221</a></li>
<li id="r15">Kebschull M, Demmer RT, Grun B, Guarnieri P, Pavlidis P, Papapanou PN. Gingival tissue transcriptomes identify distinct periodontitis phenotypes. <em>J Dent Res</em>. 2014;93(5):459-468. <a href="https://doi.org/10.1177/0022034514527288">doi:10.1177/0022034514527288</a></li>
<li id="r16">Demmer RT, Behle JH, Wolf DL, et al. Transcriptomes in healthy and diseased gingival tissues. <em>J Periodontol</em>. 2008;79(11):2112-2124. <a href="https://doi.org/10.1902/jop.2008.080139">doi:10.1902/jop.2008.080139</a></li>
<li id="r17">Irizarry RA, Hobbs B, Collin F, et al. Exploration, normalization, and summaries of high density oligonucleotide array probe level data. <em>Biostatistics</em>. 2003;4(2):249-264. <a href="https://doi.org/10.1093/biostatistics/4.2.249">doi:10.1093/biostatistics/4.2.249</a></li>
<li id="r18">Ashburner M, Ball CA, Blake JA, et al. Gene Ontology: tool for the unification of biology. <em>Nat Genet</em>. 2000;25(1):25-29. <a href="https://doi.org/10.1038/75556">doi:10.1038/75556</a></li>
<li id="r19">The Gene Ontology Consortium. The Gene Ontology knowledgebase in 2023. <em>Genetics</em>. 2023;224(1):iyad031. <a href="https://doi.org/10.1093/genetics/iyad031">doi:10.1093/genetics/iyad031</a></li>
<li id="r20">Kanehisa M, Goto S. KEGG: Kyoto Encyclopedia of Genes and Genomes. <em>Nucleic Acids Res</em>. 2000;28(1):27-30. <a href="https://doi.org/10.1093/nar/28.1.27">doi:10.1093/nar/28.1.27</a></li>
<li id="r21">Chen EY, Tan CM, Kou Y, et al. Enrichr: interactive and collaborative HTML5 gene list enrichment analysis tool. <em>BMC Bioinformatics</em>. 2013;14:128. <a href="https://doi.org/10.1186/1471-2105-14-128">doi:10.1186/1471-2105-14-128</a></li>
<li id="r22">Kuleshov MV, Jones MR, Rouillard AD, et al. Enrichr: a comprehensive gene set enrichment analysis web server 2016 update. <em>Nucleic Acids Res</em>. 2016;44(W1):W90-W97. <a href="https://doi.org/10.1093/nar/gkw377">doi:10.1093/nar/gkw377</a></li>
<li id="r23">Goeman JJ, Buhlmann P. Analyzing gene expression data in terms of gene sets: methodological issues. <em>Bioinformatics</em>. 2007;23(8):980-987. <a href="https://doi.org/10.1093/bioinformatics/btm051">doi:10.1093/bioinformatics/btm051</a></li>
<li id="r24">Simon R, Radmacher MD, Dobbin K, McShane LM. Pitfalls in the use of DNA microarray data for diagnostic and prognostic classification. <em>J Natl Cancer Inst</em>. 2003;95(1):14-18. <a href="https://doi.org/10.1093/jnci/95.1.14">doi:10.1093/jnci/95.1.14</a></li>
<li id="r25">Ioannidis JPA, Allison DB, Ball CA, et al. Repeatability of published microarray gene expression analyses. <em>Nat Genet</em>. 2009;41(2):149-155. <a href="https://doi.org/10.1038/ng.295">doi:10.1038/ng.295</a></li>
<li id="r26">Wu D, Smyth GK. Camera: a competitive gene set test accounting for inter-gene correlation. <em>Nucleic Acids Res</em>. 2012;40(17):e133. <a href="https://doi.org/10.1093/nar/gks461">doi:10.1093/nar/gks461</a></li>
<li id="r27">Conesa A, Madrigal P, Tarazona S, et al. A survey of best practices for RNA-seq data analysis. <em>Genome Biol</em>. 2016;17:13. <a href="https://doi.org/10.1186/s13059-016-0881-8">doi:10.1186/s13059-016-0881-8</a></li>
<li id="r28">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r29">Benjamini Y, Yekutieli D. The control of the false discovery rate in multiple testing under dependency. <em>Ann Stat</em>. 2001;29(4):1165-1188. <a href="https://doi.org/10.1214/aos/1013699998">doi:10.1214/aos/1013699998</a></li>
<li id="r30">Korotkevich G, Sukhov V, Budin N, Shpak B, Artyomov MN, Sergushichev A. Fast gene set enrichment analysis. <em>bioRxiv</em>. 2021:060012. <a href="https://doi.org/10.1101/060012">doi:10.1101/060012</a></li>
<li id="r31">Wright MN, Ziegler A. ranger: a fast implementation of random forests for high dimensional data in C++ and R. <em>J Stat Softw</em>. 2017;77(1):1-17. <a href="https://doi.org/10.18637/jss.v077.i01">doi:10.18637/jss.v077.i01</a></li>
<li id="r32">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
<li id="r33">The UniProt Consortium. UniProt: the Universal Protein Knowledgebase in 2023. <em>Nucleic Acids Res</em>. 2023;51(D1):D523-D531. <a href="https://doi.org/10.1093/nar/gkac1052">doi:10.1093/nar/gkac1052</a></li>
<li id="r34">Fu L, Niu B, Zhu Z, Wu S, Li W. CD-HIT: accelerated for clustering the next-generation sequencing data. <em>Bioinformatics</em>. 2012;28(23):3150-3152. <a href="https://doi.org/10.1093/bioinformatics/bts565">doi:10.1093/bioinformatics/bts565</a></li>
<li id="r35">Suzek BE, Wang Y, Huang H, McGarvey PB, Wu CH. UniRef clusters: a comprehensive and scalable alternative for improving sequence similarity searches. <em>Bioinformatics</em>. 2015;31(6):926-932. <a href="https://doi.org/10.1093/bioinformatics/btu739">doi:10.1093/bioinformatics/btu739</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. - Chapter 18 on the p &gt;&gt; n problem and on multiple testing.</li>
<li>Holmes S, Huber W. <em>Modern Statistics for Modern Biology</em>. Cambridge University Press; 2019. - High-throughput data, testing and multiple comparisons, with R code, freely available online.</li>
<li>Wasserman L. <em>All of Statistics</em>. Springer; 2004. - Chapters 10 and 15 for the testing background.</li>
<li>Gentleman R, Carey V, Huber W, Irizarry R, Dudoit S. <em>Bioinformatics and Computational Biology Solutions Using R and Bioconductor</em>. Springer; 2005. - The design philosophy behind limma and the Bioconductor data structures.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>The limma user's guide, the reference for design matrices and contrasts: <a href="https://bioconductor.org/packages/limma/">bioconductor.org/packages/limma</a>.</li>
<li><code>fgsea</code> vignette and algorithm description: <a href="https://bioconductor.org/packages/fgsea/">bioconductor.org/packages/fgsea</a>.</li>
<li>UniProt, the source of the frozen family set, and its UniRef clustered views: <a href="https://www.uniprot.org/">uniprot.org</a>.</li>
<li>GSEA and MSigDB documentation: <a href="https://www.gsea-msigdb.org/gsea/index.jsp">gsea-msigdb.org</a>.</li>
<li>Enrichr gene-set libraries (the source of the frozen GMT files): <a href="https://maayanlab.cloud/Enrichr/">maayanlab.cloud/Enrichr</a>.</li>
<li>NCBI GEO, where the series analysed here is archived: <a href="https://www.ncbi.nlm.nih.gov/geo/">ncbi.nlm.nih.gov/geo</a>.</li>
</ul>

